# NIH Chest X-Ray 14-Disease Multi-Label Classification
## Research-Grade Pipeline

**Paper Narrative**: Long-tailed multi-label CXR classification via imbalance-aware training (W-BCE → ASL → LDAM-BCE ablation), domain-adapted encoder initialization (ImageNet → DINOv2 → MedCLIP), label dependency modeling (real GCNConv with empirical co-occurrence adjacency), and calibrated cross-domain evaluation on CheXpert.

### Disease Classes (14)
`Atelectasis` · `Cardiomegaly` · `Consolidation` · `Edema` · `Effusion` · `Emphysema` · `Fibrosis` · `Hernia` · `Infiltration` · `Mass` · `Nodule` · `Pleural_Thickening` · `Pneumonia` · `Pneumothorax`

| Phase | Description |
|-------|-------------|
| 0 | Setup & Dependencies |
| 1 | Dataset Download & EDA |
| 2 | Label Parsing & Multi-Hot Encoding |
| 3 | Patient-Wise Split + Zero-Leakage Proof |
| 4 | Class Imbalance Analysis & Weighted Sampler |
| 5 | Loss Functions (W-BCE / ASL / LDAM-BCE) |
| 6 | Model Architecture (ConvNeXt-L + GCNConv) |
| 7 | Data Generators (CLAHE + Augmentation) |
| 8 & 9 | Two-Phase Training (Frozen -> Fine-Tune + DRW) |
| 10 | Post-Hoc Calibration (Isotonic + ECE + Brier) |
| 11 | Threshold Optimization & Evaluation |
| 12 | Ablation / Model Comparison Table |
| 13 | Statistical Significance (Bootstrap CI + McNemar) |
| 14 | Error Analysis (FP/FN + GradCAM on Failures) |
| 15 | Uncertainty Estimation (MC Dropout) |
| 16 | Robustness Experiments |
| 17 | GradCAM Visualization |
| 18 | External Validation (CheXpert) |
| Appendix | Model Ensemble |

## PHASE 0 — Setup & Dependencies
Install all required packages. Core scientific stack, PyTorch, and `torch_geometric` for real graph-based label dependency modeling.

In [8]:
!pip install kagglehub[hf-datasets] datasets pandas numpy matplotlib seaborn scikit-learn opencv-python iterative-stratification

In [10]:
# pip install ipywidgets


In [ ]:
# Install torch_geometric for Label Graph Module
# Match your torch+CUDA version: https://data.pyg.org/whl/
# Example for torch 2.0 + CUDA 12.1:
# pip install torch_geometric
# pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
#   -f https://data.pyg.org/whl/torch-2.0.0+cu121.html

In [11]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import hamming_loss, f1_score, label_ranking_average_precision_score, precision_recall_curve, auc, roc_auc_score, confusion_matrix
import json
from pathlib import Path
import kagglehub

print("PyTorch Version:", torch.__version__)


PyTorch Version: 2.10.0+cu128


## PHASE 1 — Dataset Download & EDA
Downloads the NIH ChestX-ray14 dataset (112,120 frontal-view X-rays from 30,805 unique patients, 14 disease labels).
The dataset has severe class imbalance: Hernia ~0.2% vs Infiltration ~17% of images. This motivates LDAM-DRW.

In [ ]:
class Config:
    # Paths (set after dataset download)
    CSV_PATH    = None   # updated by cell above
    IMAGES_DIR  = None   # updated by cell above

    # Disease classes (NIH ChestX-ray14)
    DISEASE_CLASSES = [
        'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion',
        'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule',
        'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
    ]

    # Split ratios
    TRAIN_RATIO = 0.70
    VAL_RATIO   = 0.15
    TEST_RATIO  = 0.15

    # Training
    BATCH_SIZE  = 32
    IMG_SIZE    = 224
    EPOCHS      = 20
    RANDOM_SEED = 42

    # Loss function: 'wbce' | 'asl' | 'ldam'
    LOSS        = 'ldam'

    # Encoder initialization: 'imagenet' | 'dinov2' | 'medclip'
    ENCODER_INIT = 'imagenet'

    # Output directory
    OUTPUT_DIR  = '/kaggle/working'

    # DataLoader workers (0 for Windows compatibility)
    NUM_WORKERS = 0

    # Deferred Re-Weighting epoch
    DRW_EPOCH   = 5

In [12]:
# Download the dataset
print("Downloading dataset...")
dataset_path = kagglehub.dataset_download("khanfashee/nih-chest-x-ray-14-224x224-resized")
print("Dataset downloaded to:", dataset_path)

import glob
csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
images_dir = os.path.join(dataset_path, "images")
if not os.path.exists(images_dir):
    images_dir = dataset_path

# Look specifically for the main Data Entry CSV, fallback to index 0 if not found
CSV_PATH = next((f for f in csv_files if 'Data_Entry_2017.csv' in f), csv_files[0] if csv_files else None)
IMAGES_DIR = images_dir
print("CSV:", CSV_PATH)
print("Images Dir:", IMAGES_DIR)

# ── Assign to Config so all later cells can see these paths ──────
Config.CSV_PATH   = CSV_PATH
Config.IMAGES_DIR = IMAGES_DIR
if Config.CSV_PATH:
    print(f"Config.CSV_PATH   -> {Config.CSV_PATH}")
    print(f"Config.IMAGES_DIR -> {Config.IMAGES_DIR}")
else:
    print("[WARNING] No CSV found. Check dataset download.")

Dataset downloaded to: /kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized
CSV: /kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv
Images Dir: /kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized


## PHASE 2 — Label Parsing & Multi-Hot Encoding
**What**: Parses the pipe-delimited `Finding Labels` column into a binary matrix `y` of shape `[N_images, 14]`.
**Why**: Multi-hot encoding allows each image to carry independent binary supervision signals for all 14 diseases simultaneously, enabling multi-label training with BCE-based losses.

In [14]:
def parse_labels(label_string, disease_classes):
    if pd.isna(label_string) or label_string == 'No Finding':
        return []
    labels = str(label_string).split('|')
    return [l.strip() for l in labels if l.strip() in disease_classes]

if Config.CSV_PATH:
    df = pd.read_csv(Config.CSV_PATH)
    label_col = 'Finding Labels' if 'Finding Labels' in df.columns else df.columns[1]
    image_col = 'Image Index' if 'Image Index' in df.columns else df.columns[0]
    
    df['Disease_List'] = df[label_col].apply(lambda x: parse_labels(x, Config.DISEASE_CLASSES))
    
    def to_multihot(disease_list):
        hot = np.zeros(len(Config.DISEASE_CLASSES), dtype=np.int32)
        for disease in disease_list:
            if disease in Config.DISEASE_CLASSES:
                idx = Config.DISEASE_CLASSES.index(disease)
                hot[idx] = 1
        return hot
        
    y = np.array([to_multihot(diseases) for diseases in df['Disease_List']])
    print(f"Label matrix shape: {y.shape}")


Label matrix shape: (112120, 14)


## PHASE 3 — Patient-Wise Splitting & Zero-Leakage Verification
**Problem**: The NIH dataset contains patients with multiple X-rays. A naive image-level split causes data leakage — the same patient appears in both train and test sets, leading to inflated AUC scores that do not reflect real-world performance.

**Solution**: Group all images by `Patient ID`. Aggregate labels per patient (logical OR across their images). Run `MultilabelStratifiedShuffleSplit` on the patient pool to preserve label distributions. Map the patient split back to individual image paths.

**Verification**: We assert — and print — mathematically that `train patients INTERSECT val patients = EMPTY` and `train patients INTERSECT test patients = EMPTY`.

In [15]:
if Config.CSV_PATH:
    # 2) Patient-Level Split Strategy to prevent data leakage
    patient_col = 'Patient ID' if 'Patient ID' in df.columns else df.columns[2]
    if patient_col not in df.columns:
        df['Patient ID'] = df[image_col].apply(lambda x: x.split('_')[0])
        patient_col = 'Patient ID'
        
    patients = df[patient_col].unique()
    
    # Aggregate labels per patient using logical OR
    patient_labels = df.groupby(patient_col)['Disease_List'].apply(lambda x: list(set([item for sublist in x for item in sublist]))).reset_index()
    y_patient = np.array([to_multihot(diseases) for diseases in patient_labels['Disease_List']])
    
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=(1 - Config.TRAIN_RATIO), random_state=Config.RANDOM_SEED)
    train_p_idx, temp_p_idx = next(msss.split(patient_labels[patient_col].values, y_patient))
    
    train_patients = patient_labels[patient_col].values[train_p_idx]
    temp_patients = patient_labels[patient_col].values[temp_p_idx]
    y_temp_patient = y_patient[temp_p_idx]
    
    val_size = Config.VAL_RATIO / (Config.VAL_RATIO + Config.TEST_RATIO)
    msss_temp = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=Config.RANDOM_SEED)
    test_p_idx, val_p_idx = next(msss_temp.split(temp_patients, y_temp_patient))
    
    val_patients = temp_patients[val_p_idx]
    test_patients = temp_patients[test_p_idx]
    
    # Map back to images
    train_mask = df[patient_col].isin(train_patients)
    val_mask = df[patient_col].isin(val_patients)
    test_mask = df[patient_col].isin(test_patients)
    
    X_train, y_train = df[train_mask][image_col].values, y[train_mask]
    X_val, y_val = df[val_mask][image_col].values, y[val_mask]
    X_test, y_test = df[test_mask][image_col].values, y[test_mask]
    
    print('Patient overlap Train/Val:', len(set(train_patients).intersection(set(val_patients))))
    print('Patient overlap Train/Test:', len(set(train_patients).intersection(set(test_patients))))
    print(f'Images -> Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')


Train: 78483 | Val: 16819 | Test: 16818


In [ ]:
if Config.CSV_PATH:
    # Zero-leakage proof block
    train_set = set(train_patients)
    val_set   = set(val_patients)
    test_set  = set(test_patients)

    assert len(train_set & val_set)  == 0, "DATA LEAKAGE: train/val patient overlap!"
    assert len(train_set & test_set) == 0, "DATA LEAKAGE: train/test patient overlap!"
    assert len(val_set   & test_set) == 0, "DATA LEAKAGE: val/test patient overlap!"

    print("[OK] Zero patient overlap confirmed across all splits.")
    print(f"   Train : {len(train_set):>6,} patients  |  {len(X_train):>7,} images")
    print(f"   Val   : {len(val_set):>6,} patients  |  {len(X_val):>7,} images")
    print(f"   Test  : {len(test_set):>6,} patients  |  {len(X_test):>7,} images")
    total_p = len(train_set) + len(val_set) + len(test_set)
    total_i = len(X_train) + len(X_val) + len(X_test)
    print(f"   Total : {total_p:>6,} patients  |  {total_i:>7,} images")

## PHASE 4 — Class Imbalance Analysis & Weighted Sampler
**Why**: Chest X-ray datasets exhibit severe long-tail distributions. Without correction, models collapse to predicting only majority-class diseases (Infiltration, Effusion) while ignoring rare ones (Hernia, Pneumonia).
**Strategy**: Compute per-class positive rates. Use `WeightedRandomSampler` to oversample rare classes during training. Separately compute class weights for LDAM and W-BCE losses.

In [ ]:
if Config.CSV_PATH:
    import numpy as np

    # Compute per-class positive rate and imbalance-weighted class weights
    class_weights = {}
    pos_rates = {}
    for i, disease in enumerate(Config.DISEASE_CLASSES):
        pos_count = y_train[:, i].sum()
        neg_count = len(y_train) - pos_count
        pos_rate  = pos_count / len(y_train)
        pos_rates[disease] = pos_rate
        # Inverse frequency weight, capped at 10.0 to prevent instability
        class_weights[disease] = min(neg_count / (pos_count + 1e-6), 10.0)

    print("Class imbalance analysis (training set):")
    print(f"{'Disease':<25} {'Pos%':>6}  {'Weight':>7}")
    print("-" * 42)
    for d in Config.DISEASE_CLASSES:
        print(f"{d:<25} {pos_rates[d]*100:>5.2f}%  {class_weights[d]:>7.2f}")

## PHASE 5 — Loss Function Ablation (W-BCE vs ASL vs LDAM-BCE)
**Motivation**: Standard BCE treats all classes and samples equally. For long-tail datasets this leads to poor rare-class recall.
We implement and ablate three progressively stronger losses:

| Loss | Key Idea |
|------|----------|
| **Weighted BCE** | Per-class positive weight inversely proportional to frequency |
| **ASL** (Asymmetric Loss) | Asymmetric focusing: harder penalty on false negatives |
| **LDAM-BCE** | Label-Distribution-Aware Margin: subtracts a frequency-proportional margin from positive logits |

`Config.LOSS = 'ldam'` controls which loss is active. Change to `'wbce'` or `'asl'` to reproduce ablation rows.
This creates a clean 3-row ablation in the comparison table.

## PHASE 6 — Model Architecture: ConvNeXt-L + Label Graph Module (GCNConv)

### Backbone Options
| Backbone | Purpose |
|----------|---------|
| **DenseNet121** | Established NIH CXR baseline (Wang et al. 2017) |
| **EfficientNet-B0** | Efficient scaling comparison |
| **ConvNeXt-Large** | Modern pure-CNN, outperforms older designs |

### Encoder Initialization
| Init | Source |
|------|--------|
| **ImageNet** | Standard supervised pretraining |
| **DINOv2-Large** | Self-supervised ViT — strong general representations |
| **MedCLIP** | CLIP pretrained specifically on CXR image-report pairs |

Controlled via `Config.ENCODER_INIT = 'imagenet' | 'dinov2' | 'medclip'`.

### Label Graph Module (Real GCNConv)
Diseases co-occur non-randomly (Consolidation + Effusion; Cardiomegaly + Edema).
We compute an empirical co-occurrence adjacency matrix `A` from the training labels and run 2-layer GCN message passing:
```
logits_final = logits_backbone + GCN(logits_backbone, A_cooccurrence)
```
This residual design ensures the GCN only adds structure-aware corrections on top of the backbone predictions.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

import torch.nn.functional as F

# ──────────────────────────────────────────────────────────────────────
# PHASE 5: Loss Functions
# ──────────────────────────────────────────────────────────────────────

class WeightedBCELoss(nn.Module):
    """Baseline: BCE with per-class positive weights."""
    def __init__(self, weight_tensor):
        super().__init__()
        self.weight_tensor = weight_tensor

    def forward(self, logits, labels, weight=None):
        w = self.weight_tensor if weight is None else weight
        return F.binary_cross_entropy_with_logits(logits, labels, pos_weight=w)


class AsymmetricLoss(nn.Module):
    """ASL: Asymmetric focusing — harder penalty on FN, softer on FP."""
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, x, y, weight=None):
        x_sig = torch.clamp(torch.sigmoid(x), min=self.eps, max=1 - self.eps)
        xs_pos = x_sig
        xs_neg = (x_sig + self.clip).clamp(max=1) if self.clip > 0 else x_sig
        los_pos = y * torch.log(xs_pos) * (1 - xs_pos) ** self.gamma_pos
        los_neg = (1 - y) * torch.log(1 - xs_neg) * xs_neg ** self.gamma_neg
        loss = -(los_pos + los_neg)
        return loss.mean()


class LDAMLoss(nn.Module):
    """LDAM-BCE: Label-Distribution-Aware Margin for multi-label classification.
    
    Subtracts a class-frequency-proportional margin from positive logits:
        margin_i = C / n_i^{1/4}
    where n_i is the number of positive samples for class i.
    
    Reference: Cao et al. (NeurIPS 2019) — adapted from single-label to multi-label
    by applying the margin via BCE-with-logits rather than cross-entropy.
    """
    def __init__(self, cls_num_list, max_m=0.5, s=30):
        super().__init__()
        m_list = 1.0 / np.sqrt(np.sqrt(cls_num_list))
        m_list = m_list * (max_m / np.max(m_list))
        self.m_list = torch.tensor(m_list, dtype=torch.float32, requires_grad=False).to(device)
        self.s = s

    def forward(self, logits, labels, weight=None):
        # Subtract margin only on positive-label positions
        margin = self.m_list.unsqueeze(0) * labels
        adjusted_logits = logits - margin
        return F.binary_cross_entropy_with_logits(
            adjusted_logits * self.s, labels,
            pos_weight=weight
        )


# ──────────────────────────────────────────────────────────────────────
# PHASE 6: Label Graph Module (Real GCNConv)
# ──────────────────────────────────────────────────────────────────────

class LabelGraphModule(nn.Module):
    """Real 2-layer GCNConv label dependency module.
    
    Each disease is treated as a node in a graph.
    Node features = the backbone logit for that disease.
    Edges = empirically derived from co-occurrence adjacency matrix A.
    
    Message passing (Kipf & Welling, ICLR 2017):
        H^(l+1) = ReLU( A_hat * H^(l) * W^(l) )
    where A_hat = D^{-1/2} (A + I) D^{-1/2}  (normalized adjacency with self-loops)
    
    Final output adds the GCN-refined logits as a residual correction:
        logits_final = logits_backbone + GCN(logits_backbone)
    This ensures the GCN only adds structure-aware deltas, not replacing backbone predictions.
    """

    def __init__(self, num_classes: int = 14, hidden_dim: int = 64):
        super().__init__()
        self.num_classes = num_classes

        try:
            from torch_geometric.nn import GCNConv as PyGGCNConv
            self.use_pyg = True
            self.gcn1 = PyGGCNConv(1, hidden_dim)
            self.gcn2 = PyGGCNConv(hidden_dim, 1)
            print("[LabelGraphModule] Using torch_geometric GCNConv.")
        except ImportError:
            # Fallback: manual normalized-adjacency GCN (no extra dependency)
            self.use_pyg = False
            self.W1 = nn.Linear(1, hidden_dim, bias=False)
            self.W2 = nn.Linear(hidden_dim, 1, bias=False)
            self.adj_hat = None   # set by set_adjacency()
            print("[LabelGraphModule] torch_geometric not found. Using built-in normalized-adjacency GCN.")

        self.dropout = nn.Dropout(0.3)
        self._edge_index = None

    def set_adjacency(self, edge_index: torch.Tensor, num_nodes: int, A_dense: torch.Tensor = None):
        """Store the precomputed edge_index (and optionally the normalized dense adjacency)."""
        self._edge_index = edge_index
        if not self.use_pyg and A_dense is not None:
            # Precompute D^{-1/2} (A+I) D^{-1/2}
            A = A_dense + torch.eye(num_nodes, device=A_dense.device)
            D = A.sum(dim=1).clamp(min=1e-6)
            D_inv_sqrt = torch.diag(D ** -0.5)
            self.adj_hat = D_inv_sqrt @ A @ D_inv_sqrt  # [14, 14]

    def _gcn_forward_manual(self, x):
        """Manual sparse GCN: x shape [B, 14, 1]."""
        # x: [B, 14, 1]
        A = self.adj_hat.unsqueeze(0)         # [1, 14, 14]
        h = self.W1(x)                         # [B, 14, hidden]
        h = torch.relu(torch.bmm(A.expand(x.size(0), -1, -1), h))
        h = self.dropout(h)
        h = self.W2(h)                         # [B, 14, 1]
        return h

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        """
        Args:
            logits: [B, num_classes]  — raw backbone logits
        Returns:
            refined_logits: [B, num_classes]  — residual correction added
        """
        B, C = logits.shape

        if self.use_pyg and self._edge_index is not None:
            from torch_geometric.nn import GCNConv as PyGGCNConv
            # Treat batch as independent graphs stacked via PyG batch mechanism
            # Simple approach: loop over batch (acceptable for 14-node graphs)
            corrections = []
            for b in range(B):
                x = logits[b].unsqueeze(-1)                      # [14, 1]
                h = torch.relu(self.gcn1(x, self._edge_index))   # [14, hidden]
                h = self.dropout(h)
                h = self.gcn2(h, self._edge_index)               # [14, 1]
                corrections.append(h.squeeze(-1))
            correction = torch.stack(corrections, dim=0)          # [B, 14]

        elif not self.use_pyg and self.adj_hat is not None:
            x = logits.unsqueeze(-1)                             # [B, 14, 1]
            correction = self._gcn_forward_manual(x).squeeze(-1) # [B, 14]

        else:
            # edge_index not yet set (before training data is available)
            correction = torch.zeros_like(logits)

        return logits + correction   # residual


# ──────────────────────────────────────────────────────────────────────
# PHASE 6: MultiLabelModel with DINOv2 / MedCLIP / ImageNet init
# ──────────────────────────────────────────────────────────────────────

class MultiLabelModel(nn.Module):
    """Multi-label chest X-ray classifier.
    
    Args:
        base    (str): 'convnext' | 'densenet' | 'efficientnet'
        init    (str): 'imagenet' | 'dinov2' | 'medclip'
        num_classes (int): number of disease classes (default 14)
        use_gcn (bool): attach the LabelGraphModule for disease dependency modeling
    """

    def __init__(self, base: str = 'convnext', init: str = 'imagenet',
                 num_classes: int = 14, use_gcn: bool = True):
        super().__init__()
        self.base_name = base
        self.init_name = init
        self.use_gcn   = use_gcn

        # ── Backbone ──────────────────────────────────────────────────
        if init == 'imagenet':
            num_features = self._load_imagenet_backbone(base)

        elif init == 'dinov2':
            # DINOv2-Large from Meta (HuggingFace Hub)
            # pip install transformers
            try:
                from transformers import AutoModel
                self.backbone = AutoModel.from_pretrained(
                    "facebook/dinov2-large", output_hidden_states=False
                )
                num_features = self.backbone.config.hidden_size   # 1024
                self._dino_mode = True
                print(f"[MultiLabelModel] DINOv2-Large loaded. Feature dim: {num_features}")
            except Exception as e:
                print(f"[MultiLabelModel] DINOv2 load failed: {e}. Falling back to ImageNet ConvNeXt.")
                num_features = self._load_imagenet_backbone(base)
                self._dino_mode = False
            else:
                self._dino_mode = True

        elif init == 'medclip':
            # BiomedVLP-CXR-BERT-specialized (Microsoft, HuggingFace Hub)
            # This is a CLIP-style image+text model; we use the vision encoder only.
            # pip install transformers
            try:
                from transformers import AutoModel, AutoFeatureExtractor
                self.backbone = AutoModel.from_pretrained(
                    "microsoft/BiomedVLP-CXR-BERT-specialized"
                )
                # The vision tower of BiomedVLP is a ViT; extract it
                if hasattr(self.backbone, 'vision_model'):
                    self.backbone = self.backbone.vision_model
                    num_features = self.backbone.config.hidden_size
                else:
                    # Fallback: use the full model's pooler output
                    num_features = self.backbone.config.projection_dim
                self._medclip_mode = True
                print(f"[MultiLabelModel] MedCLIP (BiomedVLP) loaded. Feature dim: {num_features}")
            except Exception as e:
                print(f"[MultiLabelModel] MedCLIP load failed: {e}. Falling back to ImageNet ConvNeXt.")
                num_features = self._load_imagenet_backbone(base)
                self._medclip_mode = False
            else:
                self._medclip_mode = True

        else:
            raise ValueError(f"Unknown init strategy: {init}. Choose 'imagenet', 'dinov2', or 'medclip'.")

        # ── Classification Head ────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

        # ── Label Graph Module (real GCNConv) ──────────────────────────
        if use_gcn:
            self.label_graph = LabelGraphModule(num_classes=num_classes)
        else:
            self.label_graph = None

    # ── Backbone loading helpers ──────────────────────────────────────
    def _load_imagenet_backbone(self, base: str) -> int:
        """Load ImageNet-pretrained backbone. Returns feature dimension."""
        self._dino_mode   = False
        self._medclip_mode = False

        if base == 'densenet':
            self.backbone   = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
            num_features    = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()

        elif base == 'efficientnet':
            self.backbone   = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            num_features    = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Identity()

        elif base == 'convnext':
            self.backbone   = models.convnext_large(weights=models.ConvNeXt_Large_Weights.IMAGENET1K_V1)
            num_features    = self.backbone.classifier[2].in_features
            self.backbone.classifier = nn.Identity()

        else:
            raise ValueError(f"Unknown base architecture: {base}. Choose 'convnext', 'densenet', or 'efficientnet'.")

        print(f"[MultiLabelModel] {base} (ImageNet) loaded. Feature dim: {num_features}")
        return num_features

    def _extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """Run the backbone and return a [B, D] feature vector."""
        if getattr(self, '_dino_mode', False):
            # DINOv2 expects pixel_values; returns last_hidden_state [B, patches+1, D]
            out = self.backbone(pixel_values=x)
            features = out.last_hidden_state[:, 0, :]   # CLS token  [B, D]

        elif getattr(self, '_medclip_mode', False):
            # BiomedVLP vision encoder; pooler_output is [B, D]
            out = self.backbone(pixel_values=x)
            features = out.pooler_output if hasattr(out, 'pooler_output')                        else out.last_hidden_state[:, 0, :]

        else:
            features = self.backbone(x)                  # [B, D]  (Identity head removed)

        return features

    def set_graph(self, edge_index: torch.Tensor, num_nodes: int,
                  A_dense: torch.Tensor = None):
        """Pass the disease co-occurrence graph to the Label Graph Module."""
        if self.label_graph is not None:
            self.label_graph.set_adjacency(edge_index, num_nodes, A_dense)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self._extract_features(x)             # [B, D]
        logits   = self.head(features)                   # [B, 14]

        if self.label_graph is not None:
            logits = self.label_graph(logits)            # [B, 14]  residual GCN

        return logits


# ──────────────────────────────────────────────────────────────────────
# Instantiate models & loss
# ──────────────────────────────────────────────────────────────────────
if Config.CSV_PATH:
    # Primary model: ConvNeXt-Large + ImageNet init + GCNConv label graph
    # Change Config.ENCODER_INIT to 'dinov2' or 'medclip' for other ablation rows
    model_dn = MultiLabelModel(
        base='convnext',
        init=Config.ENCODER_INIT,
        num_classes=len(Config.DISEASE_CLASSES),
        use_gcn=True
    ).to(device)

    # Multi-GPU support for Kaggle T4x2
    if torch.cuda.device_count() > 1:
        print(f'Using {torch.cuda.device_count()} GPUs for DataParallel!')
        model_dn = torch.nn.DataParallel(model_dn)

    # Comparison model: EfficientNet-B0 (no GCN) for ensemble
    model_eff = MultiLabelModel(
        base='efficientnet',
        init='imagenet',
        num_classes=len(Config.DISEASE_CLASSES),
        use_gcn=False
    ).to(device)
    if torch.cuda.device_count() > 1:
        model_eff = torch.nn.DataParallel(model_eff)

    # Multi-GPU support for Kaggle T4x2
    if torch.cuda.device_count() > 1:
        print(f'Using {torch.cuda.device_count()} GPUs for DataParallel!')
        model_dn = torch.nn.DataParallel(model_dn)

    # Class counts for LDAM margin computation
    class_counts  = y_train.sum(axis=0) + 1.0
    weight_tensor = torch.tensor(
        [class_weights[d] for d in Config.DISEASE_CLASSES], dtype=torch.float32
    ).to(device)

    # Multi-GPU support for Kaggle T4x2
    if torch.cuda.device_count() > 1:
        print(f'Using {torch.cuda.device_count()} GPUs for DataParallel!')
        model_dn = torch.nn.DataParallel(model_dn)

    # Select loss based on Config.LOSS flag
    if Config.LOSS == 'wbce':
        criterion = WeightedBCELoss(weight_tensor)
        print(f"Loss: WeightedBCE")
    elif Config.LOSS == 'asl':
        criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)
        print(f"Loss: AsymmetricLoss (ASL)")
    else:  # default: ldam
        criterion = LDAMLoss(cls_num_list=class_counts)
        print(f"Loss: LDAM-BCE (DRW will activate at epoch {5})")

    print(f"Backbone : ConvNeXt-Large")
    print(f"Init     : {Config.ENCODER_INIT}")
        base_model = model_dn.module if isinstance(model_dn, torch.nn.DataParallel) else model_dn
    print(f'GCN      : {base_model.use_gcn}')
    params_m = sum(p.numel() for p in model_dn.parameters()) / 1e6
    print(f"Params   : {params_m:.1f}M")


In [ ]:
# Phase 6b: Compute Label Co-Occurrence Graph & wire into model
if Config.CSV_PATH:
    import numpy as np

    # Empirical co-occurrence matrix from training labels [14, 14]
    co_matrix = y_train.T @ y_train
    np.fill_diagonal(co_matrix, 0)                       # remove self-loops
    co_norm   = co_matrix / (co_matrix.max() + 1e-8)    # normalize [0, 1]

    threshold = 0.1
    rows, cols = np.where(co_norm > threshold)
    edge_index = torch.tensor(
        np.array([rows, cols]), dtype=torch.long
    ).to(device)

    # Dense normalized adjacency for fallback GCN
    A_dense = torch.tensor(co_norm, dtype=torch.float32).to(device)

    # Wire the disease graph into the model
    model_dn.set_graph(edge_index, num_nodes=len(Config.DISEASE_CLASSES), A_dense=A_dense)
    print(f"Label co-occurrence graph: {len(rows)} directed edges (threshold={threshold})")

    print("\nSignificant disease co-occurrence pairs:")
    for r, c in zip(rows, cols):
        if r < c:
            print(f"  {Config.DISEASE_CLASSES[r]:<22} <-> {Config.DISEASE_CLASSES[c]:<22}  strength={co_norm[r,c]:.3f}")


## PHASE 7 — Data Generators (CLAHE + Augmentation)
**CLAHE** (Contrast-Limited Adaptive Histogram Equalization) normalizes local contrast in X-ray images, enhancing subtle pathological findings that appear in low-contrast lung regions.
**Training augmentation**: random rotation +/-15 deg, horizontal flip, affine shifts — simulates real-world acquisition variability without introducing artifacts that confuse pathology detection.
**Validation/test**: only resize + normalize (no augmentation, to get unbiased evaluation).

In [18]:
from PIL import Image
import cv2
cv2.setNumThreads(0) 

class ChestXrayDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
        
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            image = np.zeros((Config.IMG_SIZE, Config.IMG_SIZE), dtype=np.uint8)
        else:
            image = cv2.resize(image, (Config.IMG_SIZE, Config.IMG_SIZE))
            
        clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
        image = clahe.apply(image)
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        
        image = Image.fromarray(image)
        
        if self.transform:
            image = self.transform(image)
            
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, label

if Config.CSV_PATH:
    train_transform = transforms.Compose([
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    import glob
    all_imgs = glob.glob(os.path.join(Config.IMAGES_DIR, "**", "*.png"), recursive=True)
    img_dict = {os.path.basename(p): p for p in all_imgs}
    
    def get_paths(filenames):
        return [img_dict.get(f, os.path.join(Config.IMAGES_DIR, f)) for f in filenames]
        
    train_paths = get_paths(X_train)
    val_paths = get_paths(X_val)
    test_paths = get_paths(X_test)
    
    train_dataset = ChestXrayDataset(train_paths, y_train, transform=train_transform)
    val_dataset = ChestXrayDataset(val_paths, y_val, transform=val_transform)
    test_dataset = ChestXrayDataset(test_paths, y_test, transform=val_transform)
    
    from torch.utils.data import WeightedRandomSampler
    
    # Calculate inverse frequencies for weights
    class_counts = y_train.sum(axis=0)
    total_samples = len(y_train)
    inv_freq = total_samples / (class_counts + 1e-6)
    
    # Per-sample weight strategy: max weight among positive labels (to avoid overweighting multi-label images)
    sample_weights = np.zeros(total_samples)
    for i in range(total_samples):
        pos_indices = np.where(y_train[i] == 1)[0]
        if len(pos_indices) > 0:
            sample_weights[i] = np.max(inv_freq[pos_indices])
        else:
            sample_weights[i] = 1.0 # Base weight for completely healthy/negative
            
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=total_samples, replacement=True)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, sampler=sampler, num_workers=Config.NUM_WORKERS, pin_memory=False)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=False)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=False)


## PHASE 8 & 9 — Two-Phase Training with Deferred Re-Weighting (DRW)

### Phase A: Frozen Backbone (Phase 8)
Train only the classification head while the backbone is frozen. Learning rate: `1e-3`.
Allows rapid convergence of the classification head before the backbone adapts. This prevents early gradient signals from corrupting ImageNet-pretrained features.

### Phase B: Full Fine-Tuning (Phase 9)
Unfreeze the entire network and fine-tune at a low learning rate `1e-5`.
At this stage, Deferred Re-Weighting (DRW) activates class-balancing weights in the LDAM loss.

### Deferred Re-Weighting (DRW)
- Epoch `0 -> drw_epoch`: Pure LDAM margin training (no reweighting). Model learns stable feature representations.
- Epoch `drw_epoch -> end`: Inject class-frequency inverse weights. Focuses learning on rare classes once representations are stable.

Mixed-precision training (`torch.amp.GradScaler`) is used throughout.

In [20]:
import time
import numpy as np
import os
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
import torch
import torch.optim as optim

try:
    from trainwatch import watch
except ImportError:
    from contextlib import contextmanager
    @contextmanager
    def watch(*args, **kwargs):
        yield

def calculate_metrics(logits, labels):
    preds = logits > 0.0
    correct = (preds == labels).float()
    acc = correct.mean().item()
    
    tp = (preds & (labels == 1)).float().sum().item()
    fp = (preds & (labels == 0)).float().sum().item()
    fn = (~preds & (labels == 1)).float().sum().item()
    
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    return acc, precision, recall

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, model_name, patience=3, resume=True):
    scaler = torch.amp.GradScaler() if device.type == 'cuda' else None
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1, min_lr=1e-6)
    
    best_val_loss = float('inf')
    start_epoch = 0
    epochs_no_improve = 0
    drw_epoch = Config.DRW_EPOCH
    
    best_model_path = os.path.join(Config.OUTPUT_DIR, f'best_{model_name}.pth')
    latest_model_path = os.path.join(Config.OUTPUT_DIR, f'latest_{model_name}.pth')

    # --- RESUME CHECKPOINT LOGIC ---
    if resume and os.path.exists(latest_model_path):
        print(f"[*] Resuming from checkpoint: {latest_model_path}")
        checkpoint = torch.load(latest_model_path, map_location=device)
        
        # Handle DataParallel state dict mismatches
        state_dict = checkpoint['model_state_dict']
        if isinstance(model, torch.nn.DataParallel) and not any(k.startswith('module.') for k in state_dict.keys()):
            state_dict = {f'module.{k}': v for k, v in state_dict.items()}
        elif not isinstance(model, torch.nn.DataParallel) and any(k.startswith('module.') for k in state_dict.keys()):
            state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
            
        model.load_state_dict(state_dict)
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if scaler is not None and 'scaler_state_dict' in checkpoint:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])
        if 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        print(f"[*] Resumed at Epoch {start_epoch + 1} with Best Val Loss {best_val_loss:.4f}")
    
    if start_epoch >= num_epochs:
        print("[*] Training already completed up to requested epochs.")
        # Load best model before returning
        if os.path.exists(best_model_path):
            state_dict = torch.load(best_model_path, map_location=device)['model_state_dict']
            if isinstance(model, torch.nn.DataParallel) and not any(k.startswith('module.') for k in state_dict.keys()):
                state_dict = {f'module.{k}': v for k, v in state_dict.items()}
            model.load_state_dict(state_dict)
        return model

    # --- TRAINING LOOP ---
    for epoch in range(start_epoch, num_epochs):
        epoch_start_time = time.time()
        model.train()
        running_loss = 0.0
        running_acc, running_prec, running_rec = 0.0, 0.0, 0.0
        
        # Apply DRW weighting if past drw_epoch
        current_weight = weight_tensor if epoch >= drw_epoch else None
            
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        
        for batch_idx, (inputs, labels) in enumerate(train_pbar):
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            
            with torch.amp.autocast(device.type):
                outputs = model(inputs)
                if isinstance(criterion, LDAMLoss):
                    loss = criterion(outputs, labels, weight=current_weight)
                else:
                    loss = criterion(outputs, labels)
                
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            
            acc, prec, rec = calculate_metrics(outputs, labels)
            
            bs = inputs.size(0)
            running_loss += loss.item() * bs
            running_acc  += acc * bs
            running_prec += prec * bs
            running_rec  += rec * bs
            
            train_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
                    
        dataset_size = len(train_loader.dataset)
        epoch_loss = running_loss / dataset_size
        epoch_acc  = running_acc / dataset_size
        epoch_prec = running_prec / dataset_size
        epoch_rec  = running_rec / dataset_size
        
        # --- VALIDATION ---
        model.eval()
        val_loss = 0.0
        val_acc, val_prec, val_rec = 0.0, 0.0, 0.0
        all_val_preds, all_val_labels = [], []
        
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
        with torch.no_grad():
            for inputs, labels in val_pbar:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                with torch.amp.autocast(device.type):
                    outputs = model(inputs)
                    if isinstance(criterion, LDAMLoss):
                        loss = criterion(outputs, labels, weight=current_weight)
                    else:
                        loss = criterion(outputs, labels)
                    
                acc, prec, rec = calculate_metrics(outputs, labels)
                
                bs = inputs.size(0)
                val_loss += loss.item() * bs
                val_acc  += acc * bs
                val_prec += prec * bs
                val_rec  += rec * bs
                
                all_val_preds.append(torch.sigmoid(outputs).cpu().numpy())
                all_val_labels.append(labels.cpu().numpy())
                
        val_size = len(val_loader.dataset)
        val_loss = val_loss / val_size
        val_acc  = val_acc / val_size
        val_prec = val_prec / val_size
        val_rec  = val_rec / val_size
        
        try:
            val_preds_cat = np.vstack(all_val_preds)
            val_labels_cat = np.vstack(all_val_labels)
            val_auc = np.mean([roc_auc_score(val_labels_cat[:, i], val_preds_cat[:, i]) for i in range(14) if len(np.unique(val_labels_cat[:, i])) > 1])
        except Exception:
            val_auc = 0.0

        scheduler.step(val_loss)
        
        epoch_time = time.time() - epoch_start_time
        print(f"Epoch {epoch+1}/{num_epochs} completed in {epoch_time//60:.0f}m {epoch_time%60:.0f}s")
        print(f"  Train -> Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f} | Prec: {epoch_prec:.4f} | Rec: {epoch_rec:.4f}")
        print(f"  Val   -> Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f} | AUC: {val_auc:.4f}")
        
        # --- SAVE CHECKPOINTS ---
        # 1. Save latest state for resumption
        checkpoint_dict = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_loss': best_val_loss
        }
        if scaler is not None:
            checkpoint_dict['scaler_state_dict'] = scaler.state_dict()
            
        torch.save(checkpoint_dict, latest_model_path)
        
        # 2. Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(checkpoint_dict, best_model_path)
            epochs_no_improve = 0
            print(f"  [*] Best model updated! (Val Loss: {val_loss:.4f})")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f'  [!] Early stopping triggered after {epochs_no_improve} epochs without improvement!')
                break
                
    # End of training: load the best weights before returning
    if os.path.exists(best_model_path):
        state_dict = torch.load(best_model_path, map_location=device)['model_state_dict']
        if isinstance(model, torch.nn.DataParallel) and not any(k.startswith('module.') for k in state_dict.keys()):
            state_dict = {f'module.{k}': v for k, v in state_dict.items()}
        model.load_state_dict(state_dict)
        print("Loaded best weights for return.")
        
    return model

if Config.CSV_PATH:
    # ---------------------------------------------------------
    # PHASE A: Frozen Backbone
    # ---------------------------------------------------------
    model_name_a = f'convnext_frozen_{Config.LOSS}'
    
    print("--- PHASE A: Frozen Backbone (ConvNeXt) ---")
    base_model = model_dn.module if isinstance(model_dn, torch.nn.DataParallel) else model_dn
    for param in base_model.backbone.parameters():
        param.requires_grad = False
    
    optimizer_dn = optim.Adam(model_dn.parameters(), lr=1e-3)
    
    with watch("Phase A: Frozen ConvNeXt"):
        model_dn = train_model(model_dn, train_loader, val_loader, criterion, optimizer_dn, Config.EPOCHS, model_name_a)


--- PHASE A: Frozen Backbone (DenseNet) ---


Epoch 1/10 [Val]: 100%|██████████| 526/526 [03:31<00:00,  2.49it/s, loss=0.0510, acc=0.9135]


Epoch 1/10 - Train Loss: 0.0844 | Acc: 0.8559 | Prec: 0.3346 | Rec: 0.1569 
               Val Loss: 0.0502 | Acc: 0.9174 | Prec: 0.2029 | Rec: 0.2148 | AUC: 0.6848
[*] Best model saved! (Val Loss: 0.0502)


Epoch 2/10 [Val]: 100%|██████████| 526/526 [01:54<00:00,  4.59it/s, loss=0.0499, acc=0.9135]


Epoch 2/10 - Train Loss: 0.0819 | Acc: 0.8614 | Prec: 0.3678 | Rec: 0.1756 
               Val Loss: 0.0492 | Acc: 0.9178 | Prec: 0.2123 | Rec: 0.2284 | AUC: 0.6969
[*] Best model saved! (Val Loss: 0.0492)


...

Epoch 8/10 - Train Loss: nan | Acc: 0.8642 | Prec: 0.3998 | Rec: 0.2056 
               Val Loss: 0.0476 | Acc: 0.9231 | Prec: 0.2401 | Rec: 0.2371 | AUC: 0.7116
[*] Best model saved! (Val Loss: 0.0476)


Epoch 9/10 [Val]: 100%|██████████| 526/526 [01:48<00:00,  4.84it/s, loss=0.0479, acc=0.9398]


Epoch 9/10 - Train Loss: nan | Acc: 0.8641 | Prec: 0.4006 | Rec: 0.2113 
               Val Loss: 0.0480 | Acc: 0.9236 | Prec: 0.2355 | Rec: 0.2262 | AUC: 0.7097


Epoch 10/10 [Val]: 100%|██████████| 526/526 [01:47<00:00,  4.87it/s, loss=0.0485, acc=0.9323]


Epoch 10/10 - Train Loss: nan | Acc: 0.8648 | Prec: 0.4060 | Rec: 0.2141 
               Val Loss: 0.0482 | Acc: 0.9171 | Prec: 0.2255 | Rec: 0.2625 | AUC: 0.7119


### Phase B: Full Fine-Tuning

In [21]:
if Config.CSV_PATH:
    model_name_b = f'convnext_finetuned_{Config.LOSS}'
    
    print("\n--- PHASE B: Fine-Tuning (ConvNeXt) ---")
    base_model = model_dn.module if isinstance(model_dn, torch.nn.DataParallel) else model_dn
    for param in base_model.backbone.parameters():
        param.requires_grad = True
        
    optimizer_dn_ft = optim.Adam(model_dn.parameters(), lr=1e-5)
    
    with watch("Phase B: Fine-Tuning ConvNeXt"):
        model_dn = train_model(model_dn, train_loader, val_loader, criterion, optimizer_dn_ft, Config.EPOCHS, model_name_b)


--- PHASE B: Fine-Tuning (DenseNet) ---


Epoch 1/10 [Val]: 100%|██████████| 526/526 [01:49<00:00,  4.82it/s, loss=0.0455, acc=0.9323]


Epoch 1/10 - Train Loss: nan | Acc: 0.8675 | Prec: 0.4381 | Rec: 0.2724 
               Val Loss: 0.0458 | Acc: 0.9239 | Prec: 0.2671 | Rec: 0.2881 | AUC: 0.7568
[*] Best model saved! (Val Loss: 0.0458)


Epoch 2/10 [Val]: 100%|██████████| 526/526 [01:49<00:00,  4.79it/s, loss=0.0446, acc=0.9060]


Epoch 2/10 - Train Loss: nan | Acc: 0.8727 | Prec: 0.4763 | Rec: 0.3259 
               Val Loss: 0.0455 | Acc: 0.9199 | Prec: 0.2721 | Rec: 0.3451 | AUC: 0.7685
[*] Best model saved! (Val Loss: 0.0455)


...

Epoch 4/10 - Train Loss: nan | Acc: 0.8771 | Prec: 0.4982 | Rec: 0.3846 
               Val Loss: nan | Acc: 0.9123 | Prec: 0.2624 | Rec: 0.3996 | AUC: 0.7734


Epoch 5/10 [Val]: 100%|██████████| 526/526 [01:48<00:00,  4.84it/s, loss=0.0428, acc=0.9211]


Epoch 5/10 - Train Loss: nan | Acc: 0.8746 | Prec: 0.4881 | Rec: 0.3822 
               Val Loss: nan | Acc: 0.9172 | Prec: 0.2707 | Rec: 0.3714 | AUC: 0.7674


Epoch 6/10 [Val]: 100%|██████████| 526/526 [01:52<00:00,  4.68it/s, loss=0.0442, acc=0.9060]


Epoch 6/10 - Train Loss: nan | Acc: 0.8734 | Prec: 0.4813 | Rec: 0.3800 
               Val Loss: nan | Acc: 0.9112 | Prec: 0.2591 | Rec: 0.4015 | AUC: 0.7685
Early stopping triggered after 6 epochs!


In [ ]:
# Helper Functions for Evaluation
if Config.CSV_PATH:
    import torch
    import numpy as np
    from sklearn.metrics import f1_score

    def get_probabilities(model, loader):
        """Extract raw sigmoid probabilities from a dataloader."""
        model.eval()
        all_probs = []
        with torch.no_grad():
            for inputs, _ in loader:
                inputs = inputs.to(device)
                
                # Guard AMP usage for CPU safety
                if device.type == 'cuda':
                    with torch.amp.autocast(device.type):
                        logits = model(inputs)
                else:
                    logits = model(inputs)
                    
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs)
        return np.vstack(all_probs)

    def find_optimal_thresholds(y_true, y_prob):
        """Find the threshold that maximizes F1 score for each disease."""
        thresholds = np.linspace(0.01, 0.99, 100)
        optimal_dict = {}
        for i, disease in enumerate(Config.DISEASE_CLASSES):
            best_f1 = 0
            best_thresh = 0.5
            for t in thresholds:
                preds = (y_prob[:, i] >= t).astype(int)
                score = f1_score(y_true[:, i], preds, zero_division=0)
                if score > best_f1:
                    best_f1 = score
                    best_thresh = t
            optimal_dict[disease] = best_thresh
        return optimal_dict

## PHASE 10 — Post-Hoc Calibration (Isotonic Regression + ECE + Brier Score)
**Why calibration matters**: A model predicting `P(Pneumonia) = 0.9` should be correct ~90% of the time. Uncalibrated models are dangerously overconfident — a critical failure mode in clinical AI.

**Method**: Fit `IsotonicRegression` per class on the validation set probabilities. Apply to test set.

**Metrics**:
- **ECE** (Expected Calibration Error): alignment between predicted confidence and empirical accuracy. Target: < 0.05.
- **Brier Score**: proper scoring rule `mean((p_hat - y)^2)` per class. Lower is better.
- **Reliability Diagram**: visual calibration check — bars should lie on the diagonal.

In [23]:
if Config.CSV_PATH:
    from sklearn.metrics import classification_report
    from sklearn.isotonic import IsotonicRegression
    import numpy as np

    # Get raw probabilities from the trained model
    val_prob = get_probabilities(model_dn, val_loader)
    y_prob   = get_probabilities(model_dn, test_loader)

    # --- Isotonic Regression Calibration ---
    print("Calibrating probabilities using Isotonic Regression...")
    calibrators    = {}
    val_prob_cal   = np.zeros_like(val_prob)   # calibrated val probs
    y_prob_cal     = np.zeros_like(y_prob)     # calibrated test probs

    for i, disease in enumerate(Config.DISEASE_CLASSES):
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(val_prob[:, i], y_val[:, i])
        calibrators[disease]  = iso
        val_prob_cal[:, i]    = iso.predict(val_prob[:, i])
        y_prob_cal[:, i]      = iso.predict(y_prob[:, i])

    print("Calibration complete.")

    # --- Per-Class Threshold Optimization on calibrated val probs ---
    optimal_thresholds_dict = find_optimal_thresholds(y_val, val_prob_cal)
    optimal_thresholds      = np.array([optimal_thresholds_dict[d] for d in Config.DISEASE_CLASSES])

    # --- Test Set Predictions & Per-Disease Report ---
    y_pred = (y_prob_cal >= optimal_thresholds).astype(int)
    y_true = y_test

    print("=" * 70)
    print("PER-DISEASE PERFORMANCE (CALIBRATED)")
    print("=" * 70)
    for i, disease in enumerate(Config.DISEASE_CLASSES):
        print(f"\n{disease} (Threshold: {optimal_thresholds_dict[disease]:.3f})")
        print(classification_report(y_true[:, i], y_pred[:, i], digits=4, zero_division=0))

Calibrating probabilities using Isotonic Regression...
Calibration complete.
PER-DISEASE PERFORMANCE (CALIBRATED)

Atelectasis (Threshold: 0.160)
              precision    recall  f1-score   support

           0     0.9385    0.8187    0.8745     15088
           1     0.2518    0.5324    0.3419      1730

    accuracy                         0.7892     16818
   macro avg     0.5952    0.6755    0.6082     16818
weighted avg     0.8679    0.7892    0.8197     16818


Cardiomegaly (Threshold: 0.160)
              precision    recall  f1-score   support

           0     0.9816    0.9870    0.9843     16402
           1     0.3446    0.2692    0.3023       416

    accuracy                         0.9693     16818
   macro avg     0.6631    0.6281    0.6433     16818
weighted avg     0.9658    0.9693    0.9674     16818


Consolidation (Threshold: 0.100)
              precision    recall  f1-score   support

           0     0.9715    0.8611    0.9130     16118
           1     0.1158 

In [ ]:
# ECE and Brier Score
if Config.CSV_PATH:
    from sklearn.metrics import brier_score_loss
    import numpy as np

    def compute_ece(probs, labels, n_bins=15):
        ece_total = 0.0
        for cls_idx in range(probs.shape[1]):
            p = probs[:, cls_idx]
            l = labels[:, cls_idx]
            bins = np.linspace(0, 1, n_bins + 1)
            cls_ece = 0.0
            for b in range(n_bins):
                mask = (p >= bins[b]) & (p < bins[b + 1])
                if mask.sum() == 0:
                    continue
                acc  = l[mask].mean()
                conf = p[mask].mean()
                cls_ece += mask.sum() * abs(acc - conf) / len(p)
            ece_total += cls_ece
        return ece_total / probs.shape[1]  # mean over classes

    ece = compute_ece(y_prob_cal, y_test)
    print(f"Expected Calibration Error (ECE): {ece:.4f}  (target < 0.05)")

    brier_scores = [
        brier_score_loss(y_test[:, i], y_prob_cal[:, i])
        for i in range(len(Config.DISEASE_CLASSES))
    ]
    print("\nBrier Score per disease class:")
    for d, b in zip(Config.DISEASE_CLASSES, brier_scores):
        print(f"  {d:<25s}: {b:.4f}")
    print(f"\nMean Brier Score : {np.mean(brier_scores):.4f}")

## PHASE 11 — Threshold Optimization & Comprehensive Evaluation
**Per-class threshold tuning**: For each of the 14 diseases, sweep thresholds on the calibrated validation probabilities to find the operating point that maximizes F1 score. This is critical because the optimal threshold differs substantially between rare and common diseases.
**Metrics**: AUROC (per-class + mean), Average Precision (mAP), F1 (macro), Sensitivity, Specificity.

In [ ]:
# Model Comparison Table — populate during training runs
results_log = {}

def log_result(run_name, auroc, map_score, f1):
    results_log[run_name] = {
        'mAUROC':    round(auroc, 4),
        'mAP':       round(map_score, 4),
        'Macro-F1':  round(f1, 4),
    }
    print(f"[Result logged] {run_name}: AUROC={auroc:.4f}  mAP={map_score:.4f}  F1={f1:.4f}")

def print_comparison_table():
    import pandas as pd
    if not results_log:
        print("No results logged yet. Call log_result() after each training run.")
        return
    df = pd.DataFrame(results_log).T
    print("\n====== Model Comparison Table ======")
    print(df.to_string())
    print("====================================")
    return df

# Usage after each training run:
# log_result('ConvNeXt_MedCLIP_LDAM_GCN', val_auc, val_map, val_f1)
# print_comparison_table()

In [ ]:
if Config.CSV_PATH:
    from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
    import numpy as np

    # --- AUROC per class ---
    auroc_per_class = {}
    for i, disease in enumerate(Config.DISEASE_CLASSES):
        if len(np.unique(y_test[:, i])) > 1:
            auroc_per_class[disease] = roc_auc_score(y_test[:, i], y_prob_cal[:, i])

    mean_auroc = np.mean(list(auroc_per_class.values()))

    # --- mAP (macro) ---
    ap_per_class = {
        d: average_precision_score(y_test[:, i], y_prob_cal[:, i])
        for i, d in enumerate(Config.DISEASE_CLASSES)
        if len(np.unique(y_test[:, i])) > 1
    }
    mean_ap = np.mean(list(ap_per_class.values()))

    # --- Macro-F1 and Micro-F1 using calibrated thresholds ---
    macro_f1 = f1_score(y_test, y_pred, average='macro',  zero_division=0)
    micro_f1 = f1_score(y_test, y_pred, average='micro',  zero_division=0)

    print("=" * 55)
    print("  PHASE 11: COMPREHENSIVE EVALUATION (CALIBRATED)")
    print("=" * 55)
    print(f"  mAUROC       : {mean_auroc:.4f}")
    print(f"  mAP          : {mean_ap:.4f}")
    print(f"  Macro-F1     : {macro_f1:.4f}")
    print(f"  Micro-F1     : {micro_f1:.4f}")
    print()

    print(f"  {'Disease':<25} {'AUROC':>6}  {'AP':>6}")
    print("  " + "-" * 40)
    for d in Config.DISEASE_CLASSES:
        auc = auroc_per_class.get(d, float('nan'))
        ap  = ap_per_class.get(d, float('nan'))
        print(f"  {d:<25} {auc:>6.4f}  {ap:>6.4f}")

    # Log to results table automatically
    log_result(
        f'ConvNeXt_{Config.ENCODER_INIT}_{Config.LOSS}_GCN',
        mean_auroc, mean_ap, macro_f1
    )

## PHASE 12 — Ablation Study & Model Comparison Table
Tracks all training run results. Each row in the table corresponds to a separate experiment with one variable changed. This is the central evidence table for the paper.

| Backbone | Init | Loss | Label Module | mAUROC | mAP | Macro-F1 |
|----------|------|------|--------------|--------|-----|----------|
| DenseNet121 | ImageNet | W-BCE | None | — | — | — |
| ConvNeXt-L | ImageNet | ASL | None | — | — | — |
| ConvNeXt-L | ImageNet | LDAM-BCE | None | — | — | — |
| ConvNeXt-L | DINOv2 | LDAM-BCE | None | — | — | — |
| ConvNeXt-L | MedCLIP | LDAM-BCE | None | — | — | — |
| ConvNeXt-L | MedCLIP | LDAM-BCE | **GCNConv** | — | — | — |
| Ensemble | MedCLIP | LDAM-BCE | GCNConv | — | — | — |

## PHASE 13 — Statistical Significance Testing
**Why**: Showing one model achieves mAUROC=0.83 vs 0.81 is meaningless without uncertainty quantification. Reviewers require statistical rigor before accepting performance claims.

**Bootstrap 95% CI**: Resample the test set 1,000 times, compute AUROC each time, report [2.5th, 97.5th] percentile as the 95% confidence interval.

**McNemar Test**: Non-parametric test comparing the disagreement patterns between two models' binary predictions on the same test set — appropriate for evaluating whether the performance difference is statistically significant.

In [ ]:
# Bootstrap 95% CI on mAUROC + McNemar Test
if Config.CSV_PATH:
    from sklearn.utils import resample
    from sklearn.metrics import roc_auc_score
    import numpy as np

    def bootstrap_auroc_ci(y_true, y_prob, n_bootstrap=1000, ci=95, seed=42):
        rng = np.random.RandomState(seed)
        aucs = []
        n = len(y_true)
        for _ in range(n_bootstrap):
            idx = rng.randint(0, n, n)
            yt, yp = y_true[idx], y_prob[idx]
            try:
                auc = np.mean([
                    roc_auc_score(yt[:, i], yp[:, i])
                    for i in range(14) if len(np.unique(yt[:, i])) > 1
                ])
                aucs.append(auc)
            except Exception:
                pass
        lo = np.percentile(aucs, (100 - ci) / 2)
        hi = np.percentile(aucs, 100 - (100 - ci) / 2)
        return float(np.mean(aucs)), lo, hi

    mean_auc, lo, hi = bootstrap_auroc_ci(y_test, y_prob_cal)
    print(f"Bootstrap mAUROC: {mean_auc:.4f}  95% CI: [{lo:.4f}, {hi:.4f}]")

    # McNemar test skeleton (run after training two models)
    # from statsmodels.stats.contingency_tables import mcnemar
    # optimal_thresh_arr = np.array(list(optimal_thresholds.values()))
    # preds_a = (y_prob_convnext > optimal_thresh_arr).astype(int)
    # preds_b = (y_prob_densenet > optimal_thresh_arr).astype(int)
    # for i, disease in enumerate(Config.DISEASE_CLASSES):
    #     b = ((preds_a[:,i] == y_test[:,i]) & (preds_b[:,i] != y_test[:,i])).sum()
    #     c = ((preds_a[:,i] != y_test[:,i]) & (preds_b[:,i] == y_test[:,i])).sum()
    #     result = mcnemar([[0, b],[c, 0]], exact=False, correction=True)
    #     print(f"{disease:<25}: McNemar p={result.pvalue:.4f}")

## PHASE 14 — Error Analysis (FP/FN Breakdown)
**Why**: A model comparison table shows *what* improved. Error analysis shows *why* the remaining errors happen. This section is what separates a publishable paper from a pure benchmark report.

**Analyses**:
1. Per-disease TP/FP/FN/TN counts with Sensitivity and Specificity
2. Most confidently wrong false positives (model overconfident where truth is negative)
3. Most confidently missed false negatives (model underconfident where truth is positive)
4. GradCAM on the worst failure cases — did the model look at the right lung region?

In [ ]:
# Error Analysis
if Config.CSV_PATH:
    import numpy as np

    def error_analysis(probs, labels, thresholds, disease_names, top_n=5):
        if isinstance(thresholds, dict):
            thresh_arr = np.array([thresholds[d] for d in disease_names])
        else:
            thresh_arr = np.array(thresholds)

        preds = (probs > thresh_arr).astype(int)

        print("=" * 65)
        print("ERROR ANALYSIS: Per-Disease FP/FN Breakdown")
        print("=" * 65)

        for i, disease in enumerate(disease_names):
            tp = int(((preds[:, i] == 1) & (labels[:, i] == 1)).sum())
            fp = int(((preds[:, i] == 1) & (labels[:, i] == 0)).sum())
            fn = int(((preds[:, i] == 0) & (labels[:, i] == 1)).sum())
            tn = int(((preds[:, i] == 0) & (labels[:, i] == 0)).sum())

            sensitivity = tp / (tp + fn + 1e-8)
            specificity = tn / (tn + fp + 1e-8)

            # Most confident FPs
            fp_idx  = np.where((preds[:, i] == 1) & (labels[:, i] == 0))[0]
            fp_conf = probs[fp_idx, i] if len(fp_idx) else np.array([])
            top_fp  = np.sort(fp_conf)[::-1][:top_n]

            # Most missed FNs (low probability on true positives)
            fn_idx  = np.where((preds[:, i] == 0) & (labels[:, i] == 1))[0]
            fn_conf = probs[fn_idx, i] if len(fn_idx) else np.array([])
            top_fn  = np.sort(fn_conf)[:top_n]

            print(f"\n  {disease}")
            print(f"    TP={tp:>5}  FP={fp:>5}  FN={fn:>5}  TN={tn:>5}")
            print(f"    Sensitivity={sensitivity:.3f}  Specificity={specificity:.3f}")
            if len(top_fp): print(f"    Top FP confidences : {[f'{c:.2f}' for c in top_fp]}")
            if len(top_fn): print(f"    Worst FN probs     : {[f'{c:.2f}' for c in top_fn]}")

    error_analysis(y_prob_cal, y_test, optimal_thresholds, Config.DISEASE_CLASSES)

## PHASE 15 — Uncertainty Estimation & Abstention (MC Dropout)
**Why**: A prediction of `P(Pneumonia) = 0.71` tells a radiologist nothing about reliability. With MC Dropout, we report `0.71 +/- 0.04`, communicating both the prediction and its stability.

**Uncertainty-Based Abstention**: A deployable medical AI should "reject" or abstain from making a decision when uncertainty is too high, routing the image to a human radiologist. We measure how much accuracy improves on the remaining cases when the most uncertain 5% of predictions are rejected.

**Method**: 
1. Keep Dropout layers active at inference time (`model.train()`). 
2. Run `n_passes=30` stochastic forward passes. 
3. Report mean (prediction) and standard deviation (uncertainty) across passes.
4. Flag high-uncertainty predictions for abstention.

In [ ]:
# Monte Carlo Dropout & Abstention Analysis
if Config.CSV_PATH:
    import torch, numpy as np
    from sklearn.metrics import accuracy_score

    def mc_predict(model, loader, n_passes=30):
        model.train()   # IMPORTANT: keeps Dropout active
        all_means, all_stds, all_labels = [], [], []
        with torch.no_grad():
            for inputs, labels in loader:
                inputs = inputs.to(device)
                batch_preds = []
                for _ in range(n_passes):
                    with torch.amp.autocast(device.type):
                        logits = model(inputs)
                    batch_preds.append(torch.sigmoid(logits).cpu().numpy())
                batch_arr = np.stack(batch_preds, axis=0)  # [n_passes, B, 14]
                all_means.append(batch_arr.mean(axis=0))
                all_stds.append(batch_arr.std(axis=0))
                all_labels.append(labels.numpy())
        return np.vstack(all_means), np.vstack(all_stds), np.vstack(all_labels)

    print("Running MC Dropout (30 passes) on test set...")
    mc_mean, mc_std, mc_labels = mc_predict(model_dn, test_loader)

    # -- Abstention Analysis (Rejecting the most uncertain 5%) --
    print("\n=== Uncertainty-Based Abstention Analysis ===")
    reject_ratio = 0.05
    
    # We analyze a specific challenging disease for abstention, e.g., Infiltration
    target_idx = Config.DISEASE_CLASSES.index('Infiltration')
    
    disease_stds   = mc_std[:, target_idx]
    disease_preds  = (mc_mean[:, target_idx] > 0.5).astype(int)
    disease_labels = mc_labels[:, target_idx]
    
    uncertainty_threshold = np.percentile(disease_stds, 100 * (1 - reject_ratio))
    
    accepted_mask = disease_stds <= uncertainty_threshold
    rejected_mask = disease_stds > uncertainty_threshold
    
    acc_all      = accuracy_score(disease_labels, disease_preds)
    acc_accepted = accuracy_score(disease_labels[accepted_mask], disease_preds[accepted_mask])
    acc_rejected = accuracy_score(disease_labels[rejected_mask], disease_preds[rejected_mask])
    
    print(f"Target Disease : Infiltration")
    print(f"Reject Ratio   : {reject_ratio*100:.1f}% of highest-uncertainty predictions")
    print(f"Base Accuracy  : {acc_all:.4f} (100% of data)")
    print(f"Accepted Acc   : {acc_accepted:.4f} (improved on remaining {100-reject_ratio*100:.1f}%)")
    print(f"Rejected Acc   : {acc_rejected:.4f} (accuracy on the routed-to-human cases)")
    
    print("\nSample predictions (first 3 test images):")
    for img_idx in range(min(3, len(mc_mean))):
        print(f"\n  Image {img_idx + 1}:")
        for i, disease in enumerate(Config.DISEASE_CLASSES):
            if mc_mean[img_idx, i] > 0.15:  # only show non-trivial predictions
                flag = " [REJECT/ABSTAIN]" if mc_std[img_idx, i] > uncertainty_threshold else ""
                print(f"    {disease:<25}: {mc_mean[img_idx,i]:.3f} +/- {mc_std[img_idx,i]:.3f}{flag}")

## PHASE 16 — Robustness Experiments
**Why**: A model that only works on clean NIH images is not deployable. Real-world X-rays come with JPEG compression artifacts, brightness variation, resolution differences, and noise from different scanners. We evaluate the model under 5 clinically-motivated corruptions and report AUROC drop — the primary robustness metric.

Good robustness is especially important for cross-institution deployment, and is increasingly a requirement for medical AI publications.

In [ ]:
# Robustness Experiments
if Config.CSV_PATH:
    import torchvision.transforms.functional as TF
    from PIL import Image, ImageFilter
    import numpy as np, io
    from sklearn.metrics import roc_auc_score
    from torch.utils.data import DataLoader
    import torch

    class CorruptedDataset:
        def __init__(self, paths, labels, transform, corrupt_fn):
            self.paths, self.labels = paths, labels
            self.transform, self.corrupt_fn = transform, corrupt_fn
        def __len__(self): return len(self.paths)
        def __getitem__(self, idx):
            img = Image.open(self.paths[idx]).convert('RGB')
            img = self.corrupt_fn(img)
            return self.transform(img), torch.tensor(self.labels[idx], dtype=torch.float32)

    def run_robustness(model, corrupt_fn, name):
        ds = CorruptedDataset(test_paths, y_test, val_transform, corrupt_fn)
        loader = DataLoader(ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)
        model.eval()
        all_preds, all_lbls = [], []
        with torch.no_grad():
            for imgs, lbls in loader:
                with torch.amp.autocast(device.type):
                    preds = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
                all_preds.append(preds)
                all_lbls.append(lbls.numpy())
        p = np.vstack(all_preds); l = np.vstack(all_lbls)
        aucs = [roc_auc_score(l[:,i], p[:,i]) for i in range(14) if len(np.unique(l[:,i]))>1]
        return float(np.mean(aucs))

    corruptions = {
        'Clean (baseline)':    lambda img: img,
        'Gaussian Blur r=2':   lambda img: img.filter(ImageFilter.GaussianBlur(radius=2)),
        'Brightness +30%':     lambda img: TF.adjust_brightness(img, 1.3),
        'Brightness -30%':     lambda img: TF.adjust_brightness(img, 0.7),
        'Low-Res (64->224)':   lambda img: img.resize((64,64)).resize((Config.IMG_SIZE, Config.IMG_SIZE)),
    }

    print("\n====== Robustness Evaluation ======")
    print(f"{'Corruption':<25}  mAUROC")
    print("-" * 40)
    baseline = None
    for name, fn in corruptions.items():
        try:
            auc = run_robustness(model_dn, fn, name)
            if baseline is None: baseline = auc
            drop = auc - baseline
            print(f"{name:<25}  {auc:.4f}  (drop: {drop:+.4f})")
        except Exception as e:
            print(f"{name:<25}  ERROR: {e}")

## PHASE 17 — GradCAM Visualization
**Why**: GradCAM provides visual evidence that the model is attending to anatomically correct regions. Without this, a high AUC could result from spurious correlations (e.g., pacemaker artifacts, patient labels on films).

For ConvNeXt-Large, the target layer is: `model.backbone.features[-1]`
For DenseNet121, the target layer is: `model.backbone.features.denseblock4`

For a publishable paper, run GradCAM on failure cases identified in Phase 14 — showing *where* the model looks when it makes mistakes is a powerful qualitative contribution.

In [ ]:
# GradCAM implementation
# Adapted for ConvNeXt-Large (target: model.backbone.features[-1])
# Uses register_full_backward_hook (PyTorch >= 1.8) — avoids deprecated
# register_backward_hook which has known issues with non-tensor outputs.
import cv2, torch, numpy as np
from PIL import Image
import matplotlib.pyplot as plt


class GradCAM:
    """Gradient-weighted Class Activation Mapping.

    Uses register_full_backward_hook instead of the deprecated
    register_backward_hook. The full hook signature receives
    (module, grad_input, grad_output) where each element is a
    tuple of tensors (or None), which is more reliable across
    all module types including BatchNorm and ConvNeXt stages.
    """

    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None

        # Forward hook: save layer output (activations)
        self._fwd_hook = target_layer.register_forward_hook(
            self._save_activation
        )

        # Full backward hook: save gradient w.r.t. layer output
        # grad_output is a tuple; element [0] is the gradient tensor.
        self._bwd_hook = target_layer.register_full_backward_hook(
            self._save_gradient
        )

    # ── Hooks ────────────────────────────────────────────────────────
    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        # grad_output[0]: gradient of loss w.r.t. this layer's output
        self.gradients = grad_output[0].detach()

    # ── Cleanup ──────────────────────────────────────────────────────
    def remove(self):
        """Remove hooks to prevent memory leaks after use."""
        self._fwd_hook.remove()
        self._bwd_hook.remove()

    # ── CAM Generation ───────────────────────────────────────────────
    def generate(self, input_tensor, class_idx):
        """Produce the GradCAM heatmap for a given class index.

        Args:
            input_tensor: [1, C, H, W] preprocessed image tensor (requires_grad=True)
            class_idx   : int — disease class index to explain

        Returns:
            cam: np.ndarray [H_feat, W_feat], values in [0, 1]
        """
        self.model.eval()
        output = self.model(input_tensor)        # forward pass
        self.model.zero_grad()

        # Scalar target: sum of class_idx logits (works for batch size 1)
        target = output[:, class_idx].sum()
        target.backward()                        # backward pass

        # Global-average-pool the gradients over spatial dims → weights
        # activations: [1, C, H, W]   gradients: [1, C, H, W]
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # [1, C, 1, 1]
        cam     = (weights * self.activations).sum(dim=1, keepdim=True)
        cam     = torch.relu(cam)                # keep only positive influence
        cam     = cam.squeeze().cpu().numpy()    # [H, W]

        # Normalize to [0, 1]
        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = np.zeros_like(cam)

        return cam


def visualize_gradcam(model, img_path, disease_idx, disease_name, device,
                      alpha_heatmap=0.5):
    """Overlay GradCAM on a chest X-ray and display TP / FP / FN context.

    Args:
        model        : trained MultiLabelModel
        img_path     : str — full path to the image file
        disease_idx  : int — index in Config.DISEASE_CLASSES
        disease_name : str — label for the plot title
        device       : torch.device
        alpha_heatmap: float — heatmap blend weight (default 0.5)
    """
    from torchvision import transforms

    transform = transforms.Compose([
        transforms.Resize((Config.IMG_SIZE, Config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    img_orig   = Image.open(img_path).convert('RGB').resize(
        (Config.IMG_SIZE, Config.IMG_SIZE)
    )
    img_tensor = transform(img_orig).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # Target layer: last stage of ConvNeXt-Large backbone
    # For DenseNet121: model.backbone.features.denseblock4
    # For EfficientNet: model.backbone.features[-1]
    target_layer = model.backbone.features[-1]

    gcam = GradCAM(model, target_layer)
    cam  = gcam.generate(img_tensor, disease_idx)
    gcam.remove()   # always clean up hooks

    # Resize CAM to image size and build colour overlay
    cam_resized = cv2.resize(cam, (Config.IMG_SIZE, Config.IMG_SIZE))
    heatmap     = cv2.applyColorMap(
        np.uint8(255 * cam_resized), cv2.COLORMAP_JET
    )
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = ((1 - alpha_heatmap) * np.array(img_orig) +
               alpha_heatmap * heatmap).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(img_orig)
    axes[0].set_title('Original X-Ray')
    axes[0].axis('off')

    axes[1].imshow(cam_resized, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title(f'GradCAM: {disease_name}')
    axes[1].axis('off')

    axes[2].imshow(overlay)
    axes[2].set_title('Overlay')
    axes[2].axis('off')

    plt.suptitle(f'Disease: {disease_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Example usage (run after Phase 14 error analysis to visualise failure cases):
# visualize_gradcam(
#     model_dn,
#     test_paths[0],
#     Config.DISEASE_CLASSES.index('Effusion'),
#     'Effusion',
#     device
# )

## PHASE 18 — Domain Generalization (CheXpert & MIMIC-CXR)
**Why this matters for publication**: A model trained only on NIH may have learned dataset-specific shortcuts. True generalization requires zero-shot evaluation on data from completely different institutions (Stanford CheXpert, BIDMC MIMIC-CXR). 

A cross-dataset table showing NIH → CheXpert → MIMIC performance is one of the strongest claims in a medical AI paper.

### Cross-Dataset Label Mapping
Datasets use different terminologies. We map them strictly to NIH labels. Unmapped labels are excluded from evaluation.
- **CheXpert**: `Pleural Effusion` → `Effusion`, `Lung Opacity` → `Infiltration`, `Lung Lesion` → `Mass`, `Pleural Other` → `Pleural_Thickening`.
- **MIMIC-CXR**: Follows CheXpert labeling schema identically.
- **Handling**: Uncertain (-1) and Unmentioned (NaN) are conservatively mapped to Negative (0) to ensure strict zero-shot evaluation.

To enable: set the CSV and Image Directory paths below for your local downloads.

In [ ]:
# External Validation: CheXpert & MIMIC-CXR
# CheXpert Download: https://stanfordmlgroup.github.io/competitions/chexpert/
# MIMIC-CXR Download: https://physionet.org/content/mimic-cxr-jpg/

CHEXPERT_CSV_PATH = None  # e.g. r'D:/chexpert/CheXpert-v1.0/valid.csv'
CHEXPERT_IMG_DIR  = None  # e.g. r'D:/chexpert/'

MIMIC_CSV_PATH    = None  # e.g. r'D:/mimic-cxr/mimic-cxr-2.0.0-chexpert.csv'
MIMIC_IMG_DIR     = None  # e.g. r'D:/mimic-cxr/files/'

# Label mapping: External labels -> NIH ChestX-ray14 labels
EXTERNAL_TO_NIH = {
    'Atelectasis':      'Atelectasis',
    'Cardiomegaly':     'Cardiomegaly',
    'Consolidation':    'Consolidation',
    'Edema':            'Edema',
    'Pleural Effusion': 'Effusion',
    'Pneumonia':        'Pneumonia',
    'Pneumothorax':     'Pneumothorax',
    'Lung Opacity':     'Infiltration',
    'Lung Lesion':      'Mass',
    'Pleural Other':    'Pleural_Thickening',
    # Emphysema, Fibrosis, Hernia, Nodule: skipped (not consistently present)
}

def load_external_aligned(csv_path, img_dir, nih_classes, path_col='Path'):
    import pandas as pd, numpy as np, os
    df = pd.read_csv(csv_path)
    # Map: -1=uncertain->0, NaN=unmentioned->0, 1=positive
    nih_to_ext = {v: k for k, v in EXTERNAL_TO_NIH.items()}

    paths, labels = [], []
    for _, row in df.iterrows():
        img_path = os.path.join(img_dir, str(row[path_col])) if path_col in row else None
        if img_path is None or not os.path.exists(img_path):
            continue
        label_vec = np.zeros(len(nih_classes))
        for j, nih_name in enumerate(nih_classes):
            ext_name = nih_to_ext.get(nih_name)
            if ext_name and ext_name in df.columns:
                val = row[ext_name]
                label_vec[j] = 1.0 if val == 1.0 else 0.0
        paths.append(img_path)
        labels.append(label_vec)
    return np.array(paths), np.array(labels)

def evaluate_external_dataset(name, csv_path, img_dir, path_col='Path'):
    if not csv_path or not img_dir:
        print(f"\n[SKIPPED] {name} (paths not configured)")
        return
        
    import numpy as np
    from sklearn.metrics import roc_auc_score
    from torch.utils.data import DataLoader

    print(f"\nLoading {name} validation set...")
    ext_paths, ext_labels = load_external_aligned(csv_path, img_dir, Config.DISEASE_CLASSES, path_col)
    
    if len(ext_paths) == 0:
        print(f"No valid images found for {name}. Check paths.")
        return
        
    print(f"Loaded {len(ext_paths)} images.")

    ext_ds     = ChestXrayDataset(ext_paths, ext_labels, transform=val_transform)
    ext_loader = DataLoader(ext_ds, batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=2)
    ext_probs  = get_probabilities(model_dn, ext_loader)

    ext_aucs = {}
    for i, disease in enumerate(Config.DISEASE_CLASSES):
        # Only evaluate diseases that exist in this external dataset
        if len(np.unique(ext_labels[:, i])) > 1:
            ext_aucs[disease] = roc_auc_score(ext_labels[:, i], ext_probs[:, i])

    print(f"\n====== {name} Zero-Shot Transfer ======")
    for disease, auc in sorted(ext_aucs.items(), key=lambda x: -x[1]):
        print(f"  {disease:<25}: {auc:.4f}")
    print(f"\n  mAUROC on {name}: {np.mean(list(ext_aucs.values())):.4f}")

if Config.CSV_PATH:
    evaluate_external_dataset('CheXpert', CHEXPERT_CSV_PATH, CHEXPERT_IMG_DIR, path_col='Path')
    
    # Note: MIMIC-CXR often requires constructing the absolute path differently depending on metadata format.
    # Adjust path_col or the string parsing inside load_external_aligned if using standard physionet csv format.
    evaluate_external_dataset('MIMIC-CXR', MIMIC_CSV_PATH, MIMIC_IMG_DIR, path_col='dicom_id')

## APPENDIX C — Inference Speed & Model Size
Medical AI reviewers increasingly require inference speed reporting alongside accuracy metrics. This table documents Params (M), latency per image (ms), and FPS for each backbone.

In [ ]:
# Inference Speed & Model Size Comparison
if Config.CSV_PATH:
    import time, torch
    import numpy as np

    def benchmark_model(model, input_size=(1, 3, 224, 224), n_runs=50, device=device):
        model.eval()
        dummy = torch.randn(*input_size).to(device)
        # Warm-up
        with torch.no_grad():
            for _ in range(5):
                _ = model(dummy)
        if device.type == 'cuda':
            torch.cuda.synchronize()

        times = []
        with torch.no_grad():
            for _ in range(n_runs):
                t0 = time.perf_counter()
                _ = model(dummy)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                times.append(time.perf_counter() - t0)

        mean_ms = np.mean(times) * 1000
        fps     = 1.0 / np.mean(times)
        params  = sum(p.numel() for p in model.parameters()) / 1e6
        return mean_ms, fps, params

    models_to_bench = {
        'ConvNeXt-L (primary)': model_dn,
        'EfficientNet-B0':      model_eff,
    }

    print("=" * 58)
    print(f"{'Model':<25} {'Params':>8}  {'ms/img':>8}  {'FPS':>8}")
    print("-" * 58)
    for name, mdl in models_to_bench.items():
        try:
            ms, fps, params = benchmark_model(mdl)
            print(f"{name:<25} {params:>7.1f}M  {ms:>8.2f}  {fps:>8.1f}")
        except Exception as e:
            print(f"{name:<25} ERROR: {e}")
    print("=" * 58)


## APPENDIX D — Seed Reproducibility (Mean +/- Std)
Running with a single random seed is not sufficient for a publication claim. Report mean +/- standard deviation across 3 seeds to demonstrate that improvements are stable and not artifacts of a lucky initialization.

In [ ]:
# Seed Reproducibility: run with multiple seeds, report mean +/- std
# This block defines the infrastructure. Run after training is complete.
if Config.CSV_PATH:
    import numpy as np

    # Populate this dict by running train_model() 3 times with different seeds:
    # seed_results[seed] = {'auroc': float, 'f1': float, 'map': float}
    seed_results = {
        42:  {},   # <- fill after training run with seed=42
        123: {},   # <- fill after training run with seed=123
        999: {},   # <- fill after training run with seed=999
    }

    def report_reproducibility(seed_results, metric='auroc'):
        filled = {s: r for s, r in seed_results.items() if metric in r}
        if not filled:
            print(f"No results for metric '{metric}' yet. Run training with multiple seeds.")
            return
        vals = [r[metric] for r in filled.values()]
        print(f"\nReproducibility ({metric}) across {len(vals)} seeds:")
        for seed, r in filled.items():
            print(f"  Seed {seed}: {r[metric]:.4f}")
        print(f"  Mean : {np.mean(vals):.4f}")
        print(f"  Std  : {np.std(vals):.4f}  (report as {np.mean(vals):.4f} +/- {np.std(vals):.4f})")

    # Usage:
    # seed_results[42]  = {'auroc': 0.8312, 'f1': 0.417, 'map': 0.371}
    # seed_results[123] = {'auroc': 0.8298, 'f1': 0.412, 'map': 0.368}
    # seed_results[999] = {'auroc': 0.8335, 'f1': 0.421, 'map': 0.375}
    # report_reproducibility(seed_results, 'auroc')

    print("Seed reproducibility cell ready. Populate seed_results after 3 training runs.")


## APPENDIX — AUC-Weighted Model Ensemble (Optional)
Run only when multiple backbone models are trained. Automatically weights each model's contribution by its validation AUROC score. Typically provides a 1-2% mAUROC improvement over the best single model.

## APPENDIX B — Empirical Label Correlation Heatmap
Visualizes the raw co-occurrence matrix computed in Phase 6b. Validates the disease graph structure used in the GCNConv Label Dependency Module.

## PHASE 19 — Domain Shift & Final Results Summary

Once training and evaluation across datasets are complete, document the final performance here.

### Domain Generalization Table

| Dataset | mAUROC | mAP | Macro-F1 | Calibration (ECE) |
|---|---|---|---|---|
| NIH (Internal Test) | — | — | — | — |
| CheXpert (Zero-Shot) | — | — | — | — |
| MIMIC-CXR (Zero-Shot) | — | — | — | — |

*Note: The drop from NIH to external sets measures true domain generalization.*

### Final Selected Model (The Conference Submission Configuration)

> "After evaluating multiple initializations, loss functions, and architectural components, we selected the strongest configuration for our primary evaluation."

**Final Configuration:**
- **Encoder**: ConvNeXt-Large initialized with *<DINOv2 / MedCLIP / ImageNet>*
- **Loss Strategy**: LDAM-BCE with Deferred Re-Weighting (DRW) at Epoch 5
- **Label Dependency**: 2-layer GCNConv using empirical co-occurrence edges
- **Calibration**: Isotonic Regression
- **Clinical Safety**: High-uncertainty abstention via 30-pass MC Dropout

*This configuration establishes a strong, modern baseline for long-tailed multi-label chest X-ray classification.*